In [6]:
import pandas as pd
import numpy as np
import os

# 기준 PLD dictionary
PLD = {
    "H2" : 2.89,
    "He" : 2.60,
    "N2" : 3.64,
    "O2" : 3.46,
    "Ar" : 3.40
}

# 데이터셋 디렉토리
DIRS = [
    "Ar_273K",
    "Ar_293K",
    "Ar_313K",
    "He_273K",
    "He_293K",
    "He_313K",
    "O2_293K",
    "H2_293K",
    "N2_293K",
]

INPUT_ROOT = "../DataSet"
OUTPUT_ROOT = "../DataSet_PLDSCREENED"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

for DIR in DIRS:
    gas = DIR.split("_")[0]   # ex) Ar_273K → Ar
    threshold = PLD[gas]      # 해당 가스의 PLD 기준

    input_dir = os.path.join(INPUT_ROOT, DIR)
    output_dir = os.path.join(OUTPUT_ROOT, DIR)
    os.makedirs(output_dir, exist_ok=True)

    # CSV 파일 탐색
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv")]
    print(f"[{DIR}] Found {len(files)} CSV files.")

    for f in files:
        in_path = os.path.join(input_dir, f)
        out_path = os.path.join(output_dir, f)

        # 데이터 불러오기
        df = pd.read_csv(in_path)

        if "PLD" not in df.columns:
            print(f"⚠️ {in_path}: 'PLD' column not found, skipped.")
            continue

        # PLD 스크리닝
        PLD_SCREENED_DF = df[df["PLD"] >= threshold].reset_index(drop=True)

        # 저장
        PLD_SCREENED_DF.to_csv(out_path, index=False, encoding="utf-8-sig")

        print(f"  {f}: {len(df)} → {len(PLD_SCREENED_DF)} rows after screening")


[Ar_273K] Found 12 CSV files.
  Ar_273K_Ar_273_0.01_to_Ar_273_15_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.01_to_Ar_273_1_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.01_to_Ar_273_5_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.05_to_Ar_273_15_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.05_to_Ar_273_1_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.05_to_Ar_273_5_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.1_to_Ar_273_15_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.1_to_Ar_273_1_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.1_to_Ar_273_5_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.5_to_Ar_273_15_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.5_to_Ar_273_1_dataset.csv: 7766 → 7766 rows after screening
  Ar_273K_Ar_273_0.5_to_Ar_273_5_dataset.csv: 7766 → 7766 rows after screening
[Ar_293K] Fo

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os
import itertools
import numpy as np

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logging.info(f"Current working directory: {os.path.abspath('.')}")

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_1_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_1_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_1_dataset.csv",
}

datasets = {}
for name, path in tqdm(files.items(), desc="Loading CSV files"):
    try:
        df = pd.read_csv(path)
        henry_col = [c for c in df.columns if "input" in c.lower()]
        if not henry_col:
            raise ValueError(f"No Henry column found in {path}")
        series = df[henry_col[0]].dropna().reset_index(drop=True)
        datasets[name] = np.log10(series.clip(lower=1e-12))  # log10 변환, 음수 방지
        logging.info(f"{name}: {len(series)} values loaded (log10 transformed).")
    except Exception as e:
        logging.error(f"Failed to load {path}: {e}")

# ----- 두 개씩 조합 -----
pairs = list(itertools.combinations(datasets.keys(), 2))

# ----- 공통 축 범위 계산 -----
all_values = np.concatenate(list(datasets.values()))
xmin, xmax = np.nanmin(all_values), np.nanmax(all_values)

# ----- Figure 생성 -----
fig, axes = plt.subplots(1, len(pairs), figsize=(6*len(pairs), 6), dpi=150)

if len(pairs) == 1:
    axes = [axes]  # subplot이 1개일 때 대응

for ax, (x_name, y_name) in zip(axes, pairs):
    x = datasets[x_name]
    y = datasets[y_name]
    n = min(len(x), len(y))
    x, y = x[:n], y[:n]

    # 산점도
    ax.scatter(x, y, alpha=0.5, s=10)
    ax.set_xlabel(f"log10 Henry ({x_name})")
    ax.set_ylabel(f"log10 Henry ({y_name})")

    # 공통 축 범위 적용
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(xmin, xmax)

    # 대각선 (y=x) 참조선
    ax.plot([xmin, xmax], [xmin, xmax], "r--", lw=1)

    ax.grid(alpha=0.3)
    ax.set_title(f"{x_name} vs {y_name}")

plt.tight_layout()
save_path = "./Henry_ScatterPairs_logscale.png"
plt.savefig(save_path, dpi=300)
plt.close(fig)

logging.info(f"Saved combined scatter plot: {save_path}")


2025-09-19 18:26:39,124 [INFO] Current working directory: c:\Users\PSID_PC_20\Desktop\ActiveLearning\Data_collect\DataSet_PLDSCREENED
Loading CSV files:   0%|          | 0/3 [00:00<?, ?it/s]2025-09-19 18:26:39,172 [INFO] Ar_273K: 7768 values loaded (log10 transformed).
2025-09-19 18:26:39,198 [INFO] Ar_293K: 7768 values loaded (log10 transformed).
2025-09-19 18:26:39,228 [INFO] Ar_313K: 7768 values loaded (log10 transformed).
Loading CSV files: 100%|██████████| 3/3 [00:00<00:00, 29.10it/s]
2025-09-19 18:26:40,250 [INFO] Saved combined scatter plot: ./Henry_ScatterPairs_logscale.png


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_1_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_1_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_1_dataset.csv",
}

dfs = {}
for name, path in tqdm(files.items(), desc="Loading CSV files"):
    df = pd.read_csv(path)
    fname_col = [c for c in df.columns if "file" in c.lower()][0]
    henry_col = [c for c in df.columns if "input" in c.lower()][0]
    dfs[name] = df[[fname_col, henry_col]].rename(columns={fname_col:"filename", henry_col:name})

# ----- 세 DF merge -----
merged = dfs["Ar_273K"]
for name in ["Ar_293K", "Ar_313K"]:
    merged = pd.merge(merged, dfs[name], on="filename", how="inner")

logging.info(f"After merge: {merged.shape[0]} common filenames remain")

# ----- 상대 Henry 계산 -----
merged["rel_273K"] = 1.0
merged["rel_293K"] = merged["Ar_293K"] / merged["Ar_273K"]
merged["rel_313K"] = merged["Ar_313K"] / merged["Ar_273K"]

# ----- 시각화 -----
fig, ax = plt.subplots(figsize=(14,6), dpi=150)

x = range(len(merged))
ax.plot(x, merged["rel_273K"], marker="o", linestyle="None", color="red", label="273K (baseline)")
ax.plot(x, merged["rel_293K"], marker="o", linestyle="None", color="green", label="293K/273K")
ax.plot(x, merged["rel_313K"], marker="o", linestyle="None", color="blue", label="313K/273K")

# 무지개떡 느낌: 세 값이 같은 x에 나란히 쌓이도록 offset
offset = 0.15
ax.scatter([i-offset for i in x], merged["rel_273K"], color="red", s=10)
ax.scatter([i         for i in x], merged["rel_293K"], color="green", s=10)
ax.scatter([i+offset for i in x], merged["rel_313K"], color="blue", s=10)

ax.set_ylabel("Relative Henry coefficient (to 273K)")
ax.set_xlabel("Structures (filenames)")
ax.set_title("Relative Henry coefficients vs. 273K baseline")
ax.set_xticks([])  # xtick 숨김
ax.legend()

plt.tight_layout()
save_path = "./Relative_Henry_Multibar.png"
plt.savefig(save_path, dpi=300)
plt.close(fig)

logging.info(f"Saved plot to {save_path}")


Loading CSV files: 100%|██████████| 3/3 [00:00<00:00, 42.04it/s]
2025-09-19 18:28:45,910 [INFO] After merge: 7768 common filenames remain
2025-09-19 18:28:47,110 [INFO] Saved plot to ./Relative_Henry_Multibar.png


In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_1_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_1_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_1_dataset.csv",
}

dfs = {}
for name, path in tqdm(files.items(), desc="Loading CSV files"):
    df = pd.read_csv(path)
    fname_col = [c for c in df.columns if "file" in c.lower()][0]
    henry_col = [c for c in df.columns if "input" in c.lower()][0]
    dfs[name] = df[[fname_col, henry_col]].rename(columns={fname_col: "filename", henry_col: name})

# ----- 세 DF merge -----
merged = dfs["Ar_273K"]
for name in ["Ar_293K", "Ar_313K"]:
    merged = pd.merge(merged, dfs[name], on="filename", how="inner")

logging.info(f"After merge: {merged.shape[0]} common filenames remain")

# ----- 상대 Henry 계산 (313K 기준) -----
merged["rel_313K"] = 1.0
merged["rel_273K"] = merged["Ar_273K"] / merged["Ar_313K"]
merged["rel_293K"] = merged["Ar_293K"] / merged["Ar_313K"]

# ----- 시각화 -----
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)

x = range(len(merged))
offset = 0.15  # 무지개떡 옆으로 살짝 벌리기

# Scatter 배치
ax.scatter([i - offset for i in x], merged["rel_273K"], color="red", s=10, label="273K/313K")
ax.scatter([i          for i in x], merged["rel_293K"], color="green", s=10, label="293K/313K")
ax.scatter([i + offset for i in x], merged["rel_313K"], color="blue", s=10, label="313K baseline")

ax.set_ylabel("Relative Henry coefficient (to 313K)")
ax.set_xlabel("Structures (filenames)")
ax.set_title("Relative Henry coefficients vs. 313K baseline")
ax.set_xticks([])  # filename 표시 숨김
ax.legend()

plt.tight_layout()
save_path = "./Relative_Henry_Multibar_313K.png"
plt.savefig(save_path, dpi=300)
plt.close(fig)

logging.info(f"Saved plot to {save_path}")


Loading CSV files: 100%|██████████| 3/3 [00:00<00:00, 37.85it/s]
2025-09-19 18:29:24,649 [INFO] After merge: 7768 common filenames remain
2025-09-19 18:29:25,451 [INFO] Saved plot to ./Relative_Henry_Multibar_313K.png


In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_1_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_1_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_1_dataset.csv",
}

dfs = {}
for name, path in tqdm(files.items(), desc="Loading CSV files"):
    df = pd.read_csv(path)
    fname_col = [c for c in df.columns if "file" in c.lower()][0]
    henry_col = [c for c in df.columns if "input" in c.lower()][0]
    dfs[name] = df[[fname_col, henry_col]].rename(columns={fname_col: "filename", henry_col: name})

# ----- 세 DF merge -----
merged = dfs["Ar_273K"]
for name in ["Ar_293K", "Ar_313K"]:
    merged = pd.merge(merged, dfs[name], on="filename", how="inner")

logging.info(f"After merge: {merged.shape[0]} common filenames remain")

# ----- 상대 Henry 계산 (313K 기준) -----
merged["rel_313K"] = 1.0
merged["rel_273K"] = merged["Ar_273K"] / merged["Ar_313K"]
merged["rel_293K"] = merged["Ar_293K"] / merged["Ar_313K"]

# ----- 조건부 강조 (273 < 293 < 313) -----
highlight_mask = (merged["Ar_273K"] < merged["Ar_293K"]) & (merged["Ar_293K"] < merged["Ar_313K"])
logging.info(f"Highlighted cases (increasing with T): {highlight_mask.sum()} / {len(merged)}")

# ----- 시각화 -----
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
x = range(len(merged))
offset = 0.15

# 기본 (흐릿하게)
ax.scatter([i - offset for i in x], merged["rel_273K"], color="red", s=10, alpha=0.01)
ax.scatter([i          for i in x], merged["rel_293K"], color="green", s=10, alpha=0.01)
ax.scatter([i + offset for i in x], merged["rel_313K"], color="blue", s=10, alpha=0.01)

# 강조 (굵고 진하게)
x_high = [i for i, flag in enumerate(highlight_mask) if flag]
ax.scatter([i - offset for i in x_high], merged.loc[highlight_mask, "rel_273K"],
           color="red", s=30, alpha=0.9, label="273K/313K (highlight)")
ax.scatter([i          for i in x_high], merged.loc[highlight_mask, "rel_293K"],
           color="green", s=30, alpha=0.9, label="293K/313K (highlight)")
ax.scatter([i + offset for i in x_high], merged.loc[highlight_mask, "rel_313K"],
           color="blue", s=30, alpha=0.9, label="313K baseline (highlight)")

ax.set_ylabel("Relative Henry coefficient (to 313K)")
ax.set_xlabel("Structures (filenames)")
ax.set_title("Relative Henry coefficients vs. 313K baseline\n(Only increasing with T cases highlighted)")
ax.set_xticks([])

ax.legend()
plt.tight_layout()
save_path = "./Relative_Henry_Highlight_Increasing.png"
plt.savefig(save_path, dpi=300)
plt.close(fig)

logging.info(f"Saved plot to {save_path}")


Loading CSV files: 100%|██████████| 3/3 [00:00<00:00, 36.70it/s]
2025-09-19 18:31:23,815 [INFO] After merge: 7768 common filenames remain
2025-09-19 18:31:23,818 [INFO] Highlighted cases (increasing with T): 0 / 7768
2025-09-19 18:31:24,875 [INFO] Saved plot to ./Relative_Henry_Highlight_Increasing.png


In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os
import seaborn as sns  # jointplot 편리하게 사용

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logging.info(f"Current working directory: {os.path.abspath('.')}")

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_1_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_1_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_1_dataset.csv",
}

for name, path in tqdm(files.items(), desc="Loading CSV files"):
    df = pd.read_csv(path)

    # Henry = input, 1bar adsorption = output 찾기
    henry_col = [c for c in df.columns if "input" in c.lower()][0]
    out_col   = [c for c in df.columns if "output" in c.lower() ][0]

    logging.info(f"{name}: using {henry_col} (Henry) and {out_col} (1bar adsorption)")

    # Joint Plot 생성 (로그스케일 추천)
    g = sns.jointplot(
        data=df,
        x=henry_col,
        y=out_col,
        kind="scatter",
        marginal_kws=dict(bins=50, fill=True, alpha=0.6),
        height=6
    )

    g.set_axis_labels("Henry coefficient", "1 bar adsorption uptake")
    g.fig.suptitle(f"{name}: Henry vs. 1 bar adsorption", y=1.02)

    # 로그스케일 적용 (데이터가 넓게 퍼질 수 있으므로)
    g.ax_joint.set_xscale("log")
    g.ax_joint.set_yscale("log")

    save_path = f"./JointPlot_{name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(g.fig)

    logging.info(f"Saved joint plot for {name}: {save_path}")


2025-09-19 18:32:38,640 [INFO] Current working directory: c:\Users\PSID_PC_20\Desktop\ActiveLearning\Data_collect\DataSet_PLDSCREENED
Loading CSV files:   0%|          | 0/3 [00:00<?, ?it/s]2025-09-19 18:32:38,682 [INFO] Ar_273K: using Input (Henry) and Output (1bar adsorption)
2025-09-19 18:32:40,353 [INFO] Saved joint plot for Ar_273K: ./JointPlot_Ar_273K.png
Loading CSV files:  33%|███▎      | 1/3 [00:01<00:03,  1.71s/it]2025-09-19 18:32:40,370 [INFO] Ar_293K: using Input (Henry) and Output (1bar adsorption)
2025-09-19 18:32:41,753 [INFO] Saved joint plot for Ar_293K: ./JointPlot_Ar_293K.png
Loading CSV files:  67%|██████▋   | 2/3 [00:03<00:01,  1.53s/it]2025-09-19 18:32:41,771 [INFO] Ar_313K: using Input (Henry) and Output (1bar adsorption)
2025-09-19 18:32:43,716 [INFO] Saved joint plot for Ar_313K: ./JointPlot_Ar_313K.png
Loading CSV files: 100%|██████████| 3/3 [00:05<00:00,  1.69s/it]


In [38]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_1_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_1_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_1_dataset.csv",
}

dfs = {}
for name, path in tqdm(files.items(), desc="Loading CSV files"):
    df = pd.read_csv(path)
    fname_col = [c for c in df.columns if "file" in c.lower()][0]
    # Output = 1 bar adsorption uptake
    out_col = [c for c in df.columns if "output" in c.lower() ][0]
    dfs[name] = df[[fname_col, out_col]].rename(columns={fname_col: "filename", out_col: name})

# ----- 세 DF merge -----
merged = dfs["Ar_273K"]
for name in ["Ar_293K", "Ar_313K"]:
    merged = pd.merge(merged, dfs[name], on="filename", how="inner")

logging.info(f"After merge: {merged.shape[0]} common filenames remain")

# ----- 상대 흡착량 계산 (313K 기준) -----
merged["rel_313K"] = 1.0
merged["rel_273K"] = merged["Ar_273K"] / merged["Ar_313K"]
merged["rel_293K"] = merged["Ar_293K"] / merged["Ar_313K"]

# ----- 조건부 강조 (273 < 293 < 313) -----
highlight_mask = (merged["Ar_273K"] < merged["Ar_293K"]) | (merged["Ar_293K"] < merged["Ar_313K"])
logging.info(f"Highlighted cases (uptake increasing with T): {highlight_mask.sum()} / {len(merged)}")

# ----- 시각화 -----
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
x = range(len(merged))
offset = 0.15

# 기본 (흐릿하게)
ax.scatter([i - offset for i in x], merged["rel_273K"], color="red", s=10, alpha=0.01)
ax.scatter([i          for i in x], merged["rel_293K"], color="green", s=10, alpha=0.01)
ax.scatter([i + offset for i in x], merged["rel_313K"], color="blue", s=10, alpha=0.01)

# 강조 (굵고 진하게)
x_high = [i for i, flag in enumerate(highlight_mask) if flag]
ax.scatter([i - offset for i in x_high], merged.loc[highlight_mask, "rel_273K"],
           color="red", s=30, alpha=1, label="273K/313K (highlight)")
ax.scatter([i          for i in x_high], merged.loc[highlight_mask, "rel_293K"],
           color="green", s=30, alpha=1, label="293K/313K (highlight)")
ax.scatter([i + offset for i in x_high], merged.loc[highlight_mask, "rel_313K"],
           color="blue", s=30, alpha=1, label="313K baseline (highlight)")

ax.set_ylabel("Relative uptake (to 313K)")
ax.set_xlabel("Structures (filenames)")
ax.set_title("Relative 1 bar adsorption uptake vs. 313K baseline\n(Only increasing with T cases highlighted)")
ax.set_xticks([])

ax.legend()
plt.tight_layout()
save_path = "./Relative_Uptake_Highlight_Increasing.png"
plt.savefig(save_path, dpi=300)
plt.close(fig)

logging.info(f"Saved plot to {save_path}")


Loading CSV files: 100%|██████████| 3/3 [00:00<00:00, 50.39it/s]
2025-09-19 18:58:45,151 [INFO] After merge: 7768 common filenames remain
2025-09-19 18:58:45,153 [INFO] Highlighted cases (uptake increasing with T): 148 / 7768
2025-09-19 18:58:46,088 [INFO] Saved plot to ./Relative_Uptake_Highlight_Increasing.png


In [35]:
# 강조된 케이스 filename 추출
highlighted_files = merged.loc[highlight_mask, "filename"].tolist()

logging.info(f"Highlighted filenames (count={len(highlighted_files)}):")
for fn in highlighted_files:
    logging.info(f" - {fn}")

# CSV로 저장 (선택 사항)
out_csv = "./Highlighted_Filenames.csv"
merged.loc[highlight_mask, ["filename", "Ar_273K", "Ar_293K", "Ar_313K",
                            "rel_273K", "rel_293K", "rel_313K"]].to_csv(out_csv, index=False)
logging.info(f"Highlighted filenames with values saved to {out_csv}")


2025-09-19 18:52:48,925 [INFO] Highlighted filenames (count=148):
2025-09-19 18:52:48,926 [INFO]  - acs.cgd.5b01554_VAGNUP1452791_clean
2025-09-19 18:52:48,926 [INFO]  - CAXSUR_SL
2025-09-19 18:52:48,926 [INFO]  - TEPJAB_clean
2025-09-19 18:52:48,927 [INFO]  - TILPUA_clean
2025-09-19 18:52:48,927 [INFO]  - BEFLUV_clean
2025-09-19 18:52:48,927 [INFO]  - BEFMAC_clean
2025-09-19 18:52:48,928 [INFO]  - BICPOT_clean
2025-09-19 18:52:48,929 [INFO]  - BIWSEG_ion_b
2025-09-19 18:52:48,929 [INFO]  - BOHJOZ_auto
2025-09-19 18:52:48,930 [INFO]  - BUSQIQ_clean
2025-09-19 18:52:48,930 [INFO]  - CAXTAY_clean
2025-09-19 18:52:48,930 [INFO]  - CAXTEC_clean
2025-09-19 18:52:48,930 [INFO]  - cg5010616_si_003_clean
2025-09-19 18:52:48,931 [INFO]  - cg5010616_si_007_clean
2025-09-19 18:52:48,931 [INFO]  - cm503311x_alf100K_clean
2025-09-19 18:52:48,931 [INFO]  - cm503311x_alf11K_clean
2025-09-19 18:52:48,932 [INFO]  - cm503311x_alf125K_clean
2025-09-19 18:52:48,932 [INFO]  - cm503311x_alf150K_clean
2025-0

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import logging
from tqdm import tqdm
import os

# ----- 로깅 설정 -----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# ----- 데이터 로드 -----
files = {
    "Ar_273K": "./Ar_273K/Ar_273K_Ar_273_Henry_to_Ar_273_15_dataset.csv",
    "Ar_293K": "./Ar_293K/Ar_293K_Ar_293_Henry_to_Ar_293_15_dataset.csv",
    "Ar_313K": "./Ar_313K/Ar_313K_Ar_313_Henry_to_Ar_313_15_dataset.csv",
}

dfs = {}
for name, path in tqdm(files.items(), desc="Loading CSV files"):
    df = pd.read_csv(path)
    fname_col = [c for c in df.columns if "file" in c.lower()][0]
    # Output = 1 bar adsorption uptake
    out_col = [c for c in df.columns if "output" in c.lower() ][0]
    dfs[name] = df[[fname_col, out_col]].rename(columns={fname_col: "filename", out_col: name})

# ----- 세 DF merge -----
merged = dfs["Ar_273K"]
for name in ["Ar_293K", "Ar_313K"]:
    merged = pd.merge(merged, dfs[name], on="filename", how="inner")

logging.info(f"After merge: {merged.shape[0]} common filenames remain")

# ----- 상대 흡착량 계산 (313K 기준) -----
merged["rel_313K"] = 1.0
merged["rel_273K"] = merged["Ar_273K"] / merged["Ar_313K"]
merged["rel_293K"] = merged["Ar_293K"] / merged["Ar_313K"]

# ----- 조건부 강조 (273 < 293 < 313) -----
highlight_mask = (merged["Ar_273K"] < merged["Ar_293K"]) | (merged["Ar_293K"] < merged["Ar_313K"])
logging.info(f"Highlighted cases (uptake increasing with T): {highlight_mask.sum()} / {len(merged)}")

# ----- 시각화 -----
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
x = range(len(merged))
offset = 0.15

# 기본 (흐릿하게)
ax.scatter([i - offset for i in x], merged["rel_273K"], color="red", s=10, alpha=0.01)
ax.scatter([i          for i in x], merged["rel_293K"], color="green", s=10, alpha=0.01)
ax.scatter([i + offset for i in x], merged["rel_313K"], color="blue", s=10, alpha=0.01)

# 강조 (굵고 진하게)
x_high = [i for i, flag in enumerate(highlight_mask) if flag]
ax.scatter([i - offset for i in x_high], merged.loc[highlight_mask, "rel_273K"],
           color="red", s=30, alpha=1, label="273K/313K (highlight)")
ax.scatter([i          for i in x_high], merged.loc[highlight_mask, "rel_293K"],
           color="green", s=30, alpha=1, label="293K/313K (highlight)")
ax.scatter([i + offset for i in x_high], merged.loc[highlight_mask, "rel_313K"],
           color="blue", s=30, alpha=1, label="313K baseline (highlight)")

ax.set_ylabel("Relative uptake (to 313K)")
ax.set_xlabel("Structures (filenames)")
ax.set_title("Relative 1 bar adsorption uptake vs. 313K baseline\n(Only increasing with T cases highlighted)")
ax.set_xticks([])

ax.legend()
plt.tight_layout()
save_path = "./Relative_15barUptake_Highlight_Increasing.png"
plt.savefig(save_path, dpi=300)
plt.close(fig)

logging.info(f"Saved plot to {save_path}")


Loading CSV files: 100%|██████████| 3/3 [00:00<00:00, 32.67it/s]
2025-09-19 18:58:41,121 [INFO] After merge: 7768 common filenames remain
2025-09-19 18:58:41,127 [INFO] Highlighted cases (uptake increasing with T): 9 / 7768
2025-09-19 18:58:42,166 [INFO] Saved plot to ./Relative_15barUptake_Highlight_Increasing.png
